<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2C: *Spatial Join Fire Data*
##### Version Number: 4.0
---
### Contents  
> *Fire Damage Spatial Join of Nearest Sampling Locations*\
> *Fire Ignition Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*\
> *Fire Spread Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*\
> *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from the sampling grid.
### Inputs
- Main Dataset - `samples_res.csv` 
- Wildfire Damage Data - `fires_damage.csv` (Appendix B)
- Wildfire Ignition Data - `fires_ignition.csv` (Appendix B)
- Wildfire Spread Data - `fires_spread.csv` (Appendix B)
---
### Outputs  
- `samples_fires.csv` Sampling grid dataset merged with fire damage, spread, and ignition data.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

## Data Loading

In [3]:
samples = pd.read_csv("../data/processed/samples_res.csv")
fire_damages = pd.read_csv("../data/processed/fires_damage.csv")
fire_spread = pd.read_csv('../data/processed/fires_spread.csv')
fire_ignition = pd.read_csv('../data/processed/fires_ignition.csv')

#### Subset data for faster loops

In [4]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep].copy()

## Fire Damage Spatial Join

To prepare for feature engineering, we spatially join each fire location to its nearest sampling grid centroid. This enables us to associate daily environmental conditions with each fire based on geographic proximity, rather than relying solely on large administrative boundaries.

#### Set geometries
- Sample grid ArcGIS work was performed in EPSG 3310 to preserve area. 

In [5]:
join_samples['geometry'] = [Point(xy) for xy in zip(join_samples['centroid_easting'], join_samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

In [6]:
fire_damages['geometry'] = [Point(xy) for xy in zip(fire_damages['Fire_Longitude'], fire_damages['Fire_Latitude'])]
fire_damages_gdf = gpd.GeoDataFrame(fire_damages, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
fire_damages_projected = fire_damages_gdf.to_crs(3310)

In [7]:
fire_ignition['geometry'] = [Point(xy) for xy in zip(fire_ignition['Fire_Longitude'], fire_ignition['Fire_Latitude'])]
fire_ignition_gdf = gpd.GeoDataFrame(fire_ignition, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
fire_ignition_projected = fire_ignition_gdf.to_crs(3310)

In [8]:
fire_spread['geometry'] = [Point(xy) for xy in zip(fire_spread['Centroid_x'], fire_spread['Centroid_y'])]
fire_spread_gdf = gpd.GeoDataFrame(fire_spread, geometry='geometry', crs="EPSG:3310")

### Define default buffer
- Centroids are loosely spaced 46000 meters apart. Set buffer distance a small bit above this value. This parameter is adjustable.

In [9]:
# Square Buffer
buffer_size = 23000

### Main loop of Spatial Join algorithm

All three widlfire datasets use the same algorithm.
- Loop through all dates in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move to next date
- Load grid and fire centroids associated with the current date
- Create a buffer around each around each fire for current date
- Intersection spatial join of sampling centroids and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - Total estimated cost is summed for all fires in range
- Assign samples to new dataframe

## Fire Damage Spatial Join

Outputs:
- `damage_per_day` - ratio of damage in dollars / days burned
- `total_fire_damage` - sum of all damages done with a sampling grid
- `damage_so_far` - damages caused so far in ongoing fires in each grid

In [10]:
## for sanity check post merge
premerged = samples_gdf.copy()

for dt in fire_damages_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_damages_projected[fire_damages_projected['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_gdf[samples_gdf['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each fire on current data
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered['geometry'].apply(lambda p: square_buffer(p, buffer_size))

    # Spatial join: find sample grid centroids within each fires buffered area
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')
    
    # Aggregate data per sample
    grouped = joined.groupby('grid_id').agg(
        total_fire_damage=('Estimated Damage', 'sum'),
        Damage_per_day = ('Damage per Day', 'max'),
        Damage_so_far = ('Damage_So_Far', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_gdf.loc[samples_gdf['Date'] == dt, 'damage_per_day'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['Damage_per_day']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'total_fire_damage'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['total_fire_damage']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'damage_so_far'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['Damage_so_far']).fillna(0)

In [11]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 8)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  790836


In [12]:
samples_gdf.isna().sum()

centroid_easting          0
centroid_northing         0
Date                      0
grid_id                   0
geometry                  0
damage_per_day       263612
total_fire_damage    263612
damage_so_far        263612
dtype: int64

**Note**: NA values are for buffered areas without any fires in range within the date range. Fill these values with 0. 

In [13]:
samples_gdf = samples_gdf.fillna(0)

## Fire Ignition Spatial Join

Outputs:
- `fire_count` - count of all fires in a sampling grid

In [14]:
premerged = samples_gdf.copy()

In [15]:
## Save a copy for post merge sanity check
premerged = samples_gdf.copy()

for dt in fire_ignition_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_ignition_projected[fire_ignition_projected['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_gdf[samples_gdf['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each fire
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered['geometry'].apply(lambda p: square_buffer(p, buffer_size))

    # Spatial join: find samples within each fires buffered area
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    # Aggregate data per sample
    grouped = joined.groupby('grid_id').agg(
        total_fire_count = ('Fire Name', 'count')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_gdf.loc[samples_gdf['Date'] == dt, 'fire_count'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['total_fire_count']).fillna(0)

In [16]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 8)
Merged shape:  (608880, 9)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  6136


In [17]:
samples_gdf.isna().sum()

centroid_easting        0
centroid_northing       0
Date                    0
grid_id                 0
geometry                0
damage_per_day          0
total_fire_damage       0
damage_so_far           0
fire_count           6136
dtype: int64

**Note**: NA values caused by buffered areas without any fires in range within the date range. Fill these values with 0. 

In [18]:
samples_gdf = samples_gdf.fillna(0)

## Fire Spread Spatial Join

Outputs:
- `acres` - sum of all acres burned in a sampling grid
- `acres_per_day` - amount of acres burnt / days fire burnt
- `acres_burned_so_far` - running total of acres burnt while a wildfire is burning

In [19]:
## Save a copy for post merge sanity check
premerged = samples_gdf.copy()

for dt in fire_spread_gdf['Date'].unique():
    
    # Fires on this date
    fires_today = fire_spread_gdf[fire_spread_gdf['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_gdf[samples_gdf['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each sample
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered['geometry'].apply(lambda p: square_buffer(p, buffer_size))

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    
    # Aggregate counts and total damage per sample
    grouped = joined.groupby('grid_id').agg(
        Acres=('Acres', 'sum'),
        Acres_per_day = ('Acres per Day', 'max'),
        Acres_burned_so_far = ('Acres_Burned_So_Far','sum') 
    ).fillna(0)

    # Assign values back to main dataframe
    samples_gdf.loc[samples_gdf['Date'] == dt, 'acres'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['Acres']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'acres_per_day'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['Acres_per_day']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'acres_burned_so_far'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['Acres_burned_so_far']).fillna(0)

In [20]:
post_merge_check(samples_gdf, premerged)

Premerged shape:  (608880, 9)
Merged shape:  (608880, 12)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  229392


In [21]:
samples_gdf.isna().sum()

centroid_easting           0
centroid_northing          0
Date                       0
grid_id                    0
geometry                   0
damage_per_day             0
total_fire_damage          0
damage_so_far              0
fire_count                 0
acres                  76464
acres_per_day          76464
acres_burned_so_far    76464
dtype: int64

**Note**: NA values caused by buffered areas without any widespread fires in range within the date range. Fill these values with 0. 

In [22]:
samples_gdf = samples_gdf.fillna(0)

## Merge back to main dataset

In [23]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

In [24]:
# Merge on BOTH station and date
samples_fires = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [25]:
post_merge_check(samples_fires, samples)

Premerged shape:  (608880, 68)
Merged shape:  (608880, 75)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


In [26]:
samples_gdf = samples_gdf.fillna(0)

#### Sort dataframe by date

In [27]:
## sort dataframe by date
samples_fires = samples_fires.sort_values(by='Date')

## reset index
samples_fires = samples_fires.reset_index(drop=True)

## Export File

In [28]:
samples_fires.to_csv('../data/processed/samples_fires.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
